# Test Unlimited-OCR + dots.mocr tren Colab free (T4 16GB)
Muc tieu: doi chung 2 model SOTA voi file kho cua ban (bang bieu, do thi).
Chay tung cell theo thu tu. Het 9h/session thi mo session moi (model cache tren Drive neu mount).

In [ ]:
# Cell 0: kiem tra GPU (can T4 tro len, ~15GB VRAM)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Cell 1: cai dat (chay 1 lan; ~5-10 phut)
!pip install -q transformers==4.57.1 einops addict easydict pymupdf==1.27.2.2 qwen_vl_utils matplotlib psutil
!git clone --depth 1 https://github.com/rednote-hilab/dots.mocr.git
!pip install -q -e ./dots.mocr

In [ ]:
# Cell 2: upload PDF can test (chay cell -> chon file) + tai model
from google.colab import files
up = files.upload()
PDFS = list(up.keys())
print(PDFS)
from huggingface_hub import snapshot_download
snapshot_download('baidu/Unlimited-OCR', local_dir='/content/Unlimited-OCR')
snapshot_download('rednote-hilab/dots.mocr', local_dir='/content/DotsMOCR')

In [ ]:
# Cell 3: Unlimited-OCR (transformers, che do base; DPI 150 cho VLM do hon 300)
import os, tempfile, time, fitz, torch
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained('/content/Unlimited-OCR', trust_remote_code=True)
model = AutoModel.from_pretrained('/content/Unlimited-OCR', trust_remote_code=True,
    use_safetensors=True, torch_dtype=torch.bfloat16).eval().cuda()

def pdf_to_images(pdf, dpi=150):
    doc = fitz.open(pdf); tmp = tempfile.mkdtemp(prefix='ocr_')
    mat = fitz.Matrix(dpi/72, dpi/72); paths = []
    for i, page in enumerate(doc):
        p = os.path.join(tmp, f'page_{i+1:04d}.png')
        page.get_pixmap(matrix=mat).save(p); paths.append(p)
    doc.close(); return paths

os.makedirs('/content/out_unlimited', exist_ok=True)
for pdf in PDFS:
    t = time.time()
    model.infer_multi(tok, prompt='<image>Multi page parsing.',
        image_files=pdf_to_images(pdf), output_path='/content/out_unlimited',
        image_size=1024, max_length=32768,
        no_repeat_ngram_size=35, ngram_window=1024, save_results=True)
    print(pdf, 'XONG', round(time.time()-t), 's')

In [ ]:
# Cell 4: dots.mocr (transformers backend; T4 khong co flash-attn -> patch ve eager)
!grep -rl flash_attention_2 /content/dots.mocr --include=*.py | head
!sed -i 's/flash_attention_2/eager/g' $(grep -rl flash_attention_2 /content/dots.mocr --include=*.py)
print('patch xong')

In [ ]:
# Cell 5: chay dots.mocr parse (prompt_ocr = text sach, bo header/footer)
import subprocess
for pdf in PDFS:
    print('=====', pdf)
    subprocess.run(['python3', 'dots.mocr/dots_mocr/parser.py', pdf,
        '--prompt', 'prompt_ocr', '--use_hf', 'true'], check=False)

In [ ]:
# Cell 6: so sanh + tai ket qua ve
import glob, os
from google.colab import files as dl
for md in glob.glob('/content/out_unlimited/*.md') + glob.glob('/content/dots.mocr/*.md'):
    t = open(md, encoding='utf-8').read()
    print(md, len(t), 'ky tu | bang:', t.count('<table') + t.count('| ---'))
!zip -qr /content/ocr-colab-out.zip /content/out_unlimited /content/dots.mocr/*.md
dl.download('/content/ocr-colab-out.zip')